
# MarianMT (zh→en) — Baseline TF Training (Wuxia Domain)

**TFG Hugo Silvosa – Baseline NMT (MarianMT)**  
Este cuaderno entrena y evalúa un modelo **MarianMT** (`Helsinki-NLP/opus-mt-zh-en`) usando un dataset de **wuxia** (chino->inglés) ya preparado en formato `datasets` (HF).





## 1) Preparación del entorno

In [ ]:

import os, random, math
import numpy as np

import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))



CUDA disponible: True
Número de GPUs: 1
Nombre de la GPU: NVIDIA GeForce RTX 3060


> **Requisitos del dataset**: directorio HF Datasets con *splits* `train`, `validation`, `test` y columnas `zh` (chino) y `en` (inglés):  
> `processed_data/wuxia_zh_en_clean/`

In [ ]:
# Configuración de carpetas para entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent
BASE_DIR.mkdir(exist_ok=True)

# Estructura repositorio
for sub in ["training", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Estructura creada (si no existía):")
for p in ["trainign", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())



Base: C:\Users\Usuario\Desktop\TFG\CORPUS
Estructura creada (si no existía):
 - C:\Users\Usuario\Desktop\TFG\CORPUS\trainign
 - C:\Users\Usuario\Desktop\TFG\CORPUS\models
 - C:\Users\Usuario\Desktop\TFG\CORPUS\proccesed data


## 2) Configuración

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Rutas (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"   # <- carpeta con dataset HuggingFace 
    output_dir: Path   = BASE_DIR / "models" / "marianmt_wuxia"              # <- aquí se guardan los modelos
    ckpt_dir: Path     = BASE_DIR / "checkpoints"                          # <- checkpoints durante entrenamiento
    training_dir: Path = BASE_DIR / "training"         

    # Columnas del dataset
    src_col: str = "zh"
    tgt_col: str = "en"

    # Modelo
    model_ckpt: str = "Helsinki-NLP/opus-mt-zh-en"

    # Entrenamiento
    seed: int = 42
    max_source_length: int = 128
    max_target_length: int = 128
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    early_stopping_patience: int = 3

    fraction: float = 1

cfg = Config()
print(cfg)


Config(dataset_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/processed_data/wuxia_zh_en_clean'), output_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/models/marianmt_wuxia'), ckpt_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/checkpoints'), training_dir=WindowsPath('c:/Users/Usuario/Desktop/TFG/CORPUS/training'), src_col='zh', tgt_col='en', model_ckpt='Helsinki-NLP/opus-mt-zh-en', seed=42, max_source_length=128, max_target_length=128, batch_size=16, epochs=10, learning_rate=2e-05, weight_decay=0.01, early_stopping_patience=3, fraction=1)


In [ ]:
import random, numpy as np, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)



# Semillas 
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)


if device.type == "cuda":
    torch.cuda.manual_seed_all(cfg.seed)
    # Para reproducibilidad estricta
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Semillas fijadas y backend configurado.")


Usando dispositivo: cuda
Semillas fijadas y backend configurado.


## 3) Cargar dataset (Hugging Face Datasets)

In [ ]:

from datasets import load_from_disk, DatasetDict

assert os.path.isdir(cfg.dataset_dir), f"No se encuentra el dataset en: {cfg.dataset_dir}"
raw_ds: DatasetDict = load_from_disk(cfg.dataset_dir)
print(raw_ds)

# Validar columnas
def _check_cols(ds, src_col, tgt_col, split):
    cols = ds.column_names
    assert src_col in cols and tgt_col in cols, f"El split '{split}' debe contener columnas '{src_col}' y '{tgt_col}'. Columnas: {cols}"

for split in ["train", "validation", "test"]:
    assert split in raw_ds, f"Falta el split '{split}' en el dataset."
    _check_cols(raw_ds[split], cfg.src_col, cfg.tgt_col, split)

# pruebas 
def take_fraction(ds, frac, seed=42):
    if frac >= 1.0:
        return ds
    n = max(1, int(len(ds) * frac))
    return ds.shuffle(seed=seed).select(range(n))

train_ds = take_fraction(raw_ds["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(raw_ds["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(raw_ds["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[:2])
print(f"Tam. train/val/test (fracción={cfg.fraction}):", len(train_ds), len(val_ds), len(test_ds))


DatasetDict({
    train: Dataset({
        features: ['zh', 'en'],
        num_rows: 417208
    })
    validation: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
    test: Dataset({
        features: ['zh', 'en'],
        num_rows: 52151
    })
})
{'zh': ['第章 听到白小纯的话语，看到圣皇的迟疑，邪皇这里顿时呼吸一促，他目中刹那就露出凌厉之芒，右手猛的一挥，顿时那残破的红日，骤然幻化', '尤其是看到人群内的宋缺时，神算子立刻警惕，他当年在空城，是第一个跟随白小纯的，受到了重用，如今却成为了第二个，他顿时就视宋缺为竞争对手'], 'en': ['chapter- The Saint-Emperor hesitated, and the Vile-Emperor sucked in a breath, eyes flickering with cold light as he waved his hand to summon his damaged red sun', 'That was even more the case when he noticed Song Que among Bai Xiaochun’s men, which immediately got him even more on guard. Back in Sky City, Master God-Diviner had been the first to start following Bai Xiaochun again, and it had led to incredible benefits. Now, he was only the second to join him, which put Song Que in his sights as a major rival']}
Tam. train/val/test (fracción=1): 417208 52151 52151

## 4) Cargar tokenizador y modelo MarianMT (zh->en)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(cfg.model_ckpt)


model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_ckpt)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo cargado: {cfg.model_ckpt}")
print(f"Parámetros totales: {n_params:,}")

Modelo cargado: Helsinki-NLP/opus-mt-zh-en
Parámetros totales: 77,943,296


## 5) Preprocesamiento y tokenización

In [ ]:
# tokenizar ejemplos
def preprocess_function(examples):
    # Tokenizar textos fuente y destino
    model_inputs = tokenizer(
        examples[cfg.src_col],
        max_length=cfg.max_source_length,
        padding=False,
        truncation=True
    )

    # Tokenizar target con "labels"
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples[cfg.tgt_col],
            max_length=cfg.max_target_length,
            padding=False,
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

#  tokenización a todo el dataset
tokenized_datasets = raw_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_ds["train"].column_names
)

train_ds = take_fraction(tokenized_datasets["train"], cfg.fraction, seed=cfg.seed)
val_ds   = take_fraction(tokenized_datasets["validation"], cfg.fraction, seed=cfg.seed)
test_ds  = take_fraction(tokenized_datasets["test"], cfg.fraction, seed=cfg.seed)

print(train_ds[0])


{'input_ids': [440, 4137, 7, 5563, 4067, 1049, 17398, 2709, 3273, 2, 3213, 5251, 16968, 11, 11608, 11531, 2, 30189, 16968, 4258, 6275, 142, 14088, 333, 16683, 2, 182, 5659, 80, 52849, 33153, 9656, 854, 23153, 50521, 892, 27910, 2, 10284, 1211, 15834, 11, 333, 32430, 2, 6275, 142, 1095, 10973, 6875, 11, 5925, 75, 2, 54115, 7481, 19122, 1154, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [4101, 15, 24, 7567, 15, 339, 97, 6139, 672, 45659, 104, 2, 6, 3, 1612, 11241, 15, 339, 97, 6139, 672, 56637, 10, 12, 24174, 2, 6112, 58114, 34632, 27, 8706, 1407, 33, 169, 22800, 104, 174, 2239, 8, 52294, 174, 16168, 11386, 14243, 0]}


## 6) Data collator

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Se encarga de alinear dinámicamente las secuencias y crear batches listos para el modelo
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",   
    return_tensors="pt"
)

# Ejemplo de batch
batch = data_collator([train_ds[i] for i in range(2)])
for k, v in batch.items():
    print(f"{k}: shape={v.shape}, dtype={v.dtype}")

input_ids: shape=torch.Size([2, 59]), dtype=torch.int64
attention_mask: shape=torch.Size([2, 59]), dtype=torch.int64
labels: shape=torch.Size([2, 93]), dtype=torch.int64
decoder_input_ids: shape=torch.Size([2, 93]), dtype=torch.int64


## 7) Configuración de entrenamiento 



> **Por defecto**          
> **Optimizador**: `AdamW` (con LR=2e-5, weight decay=0.01) de `Seq2SeqTrainer`   
> **Loss**: `CrossEntropyLoss` (token-level) de `AutoModelForSeq2SeqLM `





In [ ]:

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Carpeta para resultados
run_dir = cfg.output_dir
run_dir.mkdir(parents=True, exist_ok=True)

# Argumentos de entrenamiento
training_args = Seq2SeqTrainingArguments(
    output_dir=str(run_dir),
    overwrite_output_dir=True,
    eval_strategy="epoch",                  # Evaluar al final de cada epoch
    save_strategy="epoch",                  # Guardar checkpoint por epoch
    save_total_limit=3,                      # Max nº de checkpoints guardados
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    weight_decay=cfg.weight_decay,
    logging_dir=str(run_dir / "logs"),
    logging_strategy="steps",
    logging_steps=50,
    predict_with_generate=True,              # Generar secuencias en validación
    fp16=torch.cuda.is_available(),          # Precisión mixta si hay GPU
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

# Trainer para tareas Seq2Seq
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

print(" Seq2SeqTrainer configurado (PyTorch).")


 Seq2SeqTrainer configurado (PyTorch).


## 8) Entrenamiento

In [ ]:

from transformers import EarlyStoppingCallback
import json

# Añadir callback de early stopping
trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=cfg.early_stopping_patience,  # nº de evaluaciones sin mejora
    early_stopping_threshold=0.0
))

# Entrenar
train_result = trainer.train()

# Guardar modelo final y tokenizer
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

# Guardar métricas de entrenamiento
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

# Guardar configuración y resultados en JSON 
run_info = {
    "model_name": cfg.model_ckpt,
    "epochs": cfg.epochs,
    "batch_size": cfg.batch_size,
    "learning_rate": cfg.learning_rate,
    "weight_decay": cfg.weight_decay,
    "train_size": len(train_ds),
    "val_size": len(val_ds),
    "metrics": metrics
}

with open(cfg.output_dir / "run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=4, ensure_ascii=False)

print(" Entrenamiento finalizado, modelo y artefactos guardados en:", cfg.output_dir)


Epoch,Training Loss,Validation Loss
1,1.726900,1.636841
2,1.619400,1.534088
3,1.528200,1.476575
4,1.417800,1.439425
5,1.352600,1.412678
6,1.284000,1.395350
7,1.222200,1.377264
8,1.231500,1.368366
9,1.216200,1.361179
10,1.111900,1.359185


c:\Users\Usuario\miniconda3\envs\wuxia\Lib\site-packages\transformers\modeling_utils.py:3917: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[65000]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


***** train metrics *****
  epoch                    =       10.0
  total_flos               = 60353816GF
  train_loss               =     1.3947
  train_runtime            = 7:08:15.77
  train_samples_per_second =    162.364
  train_steps_per_second   =     10.148
 Entrenamiento finalizado, modelo y artefactos guardados en: c:\Users\Usuario\Desktop\TFG\CORPUS\models\marianmt_wuxia
